# WEEK 2

### Initial Setup


In [ ]:
content = ""
with open('Leadership_and_Management_Importance_in_Organisations.txt', 'r') as file:
    content = file.read()

In [ ]:
print(len(content),content[0:500],sep='\n**\n')

#### Downloading Model and imports once for all later uses

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

## Question 1

### Part A

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 80):
    strlist = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        strlist.append(text[start:end])
        start = end - overlap
    return strlist

In [ ]:
chunks = chunk_text(text=content, chunk_size=200, overlap=80)
print(len(chunks))

### Part B

In [ ]:
model

In [ ]:
#TODO 1:
embed_chunks = model.encode(chunks)
print(embed_chunks.shape)

In [ ]:
def cosine_similarity(vec1, vec2) -> float:
    dp = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return (dp / (norm1 * norm2))

In [ ]:
def retrieve(query: str, chunks: list, embed_chunks: list, top_k: int = 3) -> list[str]:
    query_embed = model.encode(query)
    # similarities = [cosine_similarity(query_embed, chunkvector) for chunkvector in embed_chunks]
    # above is handwritten and takes longer for large lists
    similarities = (util.cos_sim(query_embed, embed_chunks)[0]).numpy()
    return [chunks[x] for x in np.argsort(similarities)[::-1][:top_k]]

### Part C

~ Using are manual pipeline for the text2text generator because the `pipeline('text2text-generator, 'modelname')` and `pipeline('text-generator, 'modelname')` both very showing all kinds of errors.
~ It is beacuse my transformers version is of >5.1 in hich these functionalities have been disabled in pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
textmodel = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

In [ ]:
def generator(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = textmodel.generate(**inputs, max_new_tokens=256)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result
generator("Me: Hi,are you well?\n Friend: ")

In [ ]:

def rag_answer(query: str) -> str:
    valid_chunks = retrieve(query, chunks, embed_chunks, top_k=3)
    context = "\n\n".join(chunk for chunk in valid_chunks)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say 'I don't know'.
Context:
{context}
Question: {query}
Answer:"""
    response = generator(prompt)
    return response
print("1. ", rag_answer("What is the importance of leadership and management in organisations?"))
print("2. ", rag_answer("What is the favourite colour of the author?"))
    

## Question 2


### Part A

#### Fixed_Chunking (fixedchunks)

In [ ]:
def fixed_chunk(content: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    chunklist = []
    start = 0
    while start < len(content):
        end = start + chunk_size
        chunklist.append(content[start:end])
        start = end - overlap
    return chunklist

In [ ]:
fixedchunks = fixed_chunk(content, chunk_size=300, overlap=50)
print(len(fixedchunks),fixedchunks[0],sep='\n***\n')

#### Sentence Wise Chunking (sentencechunks)

In [ ]:
def splittext(text, delimiters): 
    result = []
    current = []
    for ch in text:
        current.append(ch)          
        if ch in delimiters:
            result.append(''.join(current))  
            current = []           
    
    if current:                     
        result.append(''.join(current))
    
    return [item for item in result if item.strip()]

In [ ]:
def sentence_chunk(content: str, splitdelimiters : set[chr]={'\n', '!', '.', '?'}, sentencegrpsize : int=5, sentenceoverlapsize : int=1):
    sentences = splittext(content, splitdelimiters)
    start = 0
    chunks = []
    while start < len(sentences) :
        chunks.append(''.join(sentences[start:(start+sentencegrpsize)]))
        start += sentencegrpsize - sentenceoverlapsize
    return chunks

In [ ]:
sentencechunks = sentence_chunk(content)
print(len(sentencechunks),sentencechunks[0],sep="\n***\n")

#### Sliding Window Chunking (slidingchunks)

In [ ]:
def sliding_chunk(content : str, window : int=400, step : int=100):
    chunklist = []
    start = 0
    while start < len(content):
        chunklist.append(content[start:start+window])
        start += step
    return chunklist

In [ ]:
slidingchunks = sliding_chunk(content, window=400, step=100)
print(len(slidingchunks),fixedchunks[0],sep='\n***\n')

#### General Stats

In [ ]:
print('TYPE\t\t\tNO.OF CHUNKS\tMAX.LENGTH\tMIN.LENGTH\tMEAN.LENGTH')
slen = [len(x) for x in fixedchunks]
print(f'Fixed Chunking\t\t{len(slen)}\t\t{max(slen)}\t\t{min(slen)}\t\t{(sum(slen)/len(slen)):.2f}')
slen = [len(x) for x in sentencechunks]
print(f'Sentence Chunking\t{len(slen)}\t\t{max(slen)}\t\t{min(slen)}\t\t{(sum(slen)/len(slen)):.2f}')
slen = [len(x) for x in slidingchunks]
print(f'Sliding WIndow Chunking\t{len(slen)}\t\t{max(slen)}\t\t{min(slen)}\t\t{(sum(slen)/len(slen)):.2f}')

### Part B

In [ ]:
embedfixed = model.encode(fixedchunks)
embedsentence = model.encode(sentencechunks)
embedsliding = model.encode(slidingchunks)

In [ ]:
## shall copy the code of retrive used above in Question 1 Part B
def retrieve(query: str, chunks: list, embed_chunks: np.ndarray, top_k: int = 3) -> list[str]:
    query_embed = model.encode(query)
    # similarities = [cosine_similarity(query_embed, chunkvector) for chunkvector in embed_chunks]
    # above is handwritten and takes longer for large lists
    similarities = (util.cos_sim(query_embed, embed_chunks)[0]).numpy()
    return [chunks[x] for x in np.argsort(similarities)[::-1][:top_k]]

In [ ]:
qa = qa_benchmark_hard = [
    (
        "How does Daniel Goleman define the specific component of emotional intelligence that prevents individuals from becoming prisoners of their feelings?", 
        "on-going inner conversation"
    ),
    (
        "What was the primary theoretical criticism raised in the 1940s against the idea that distinctive characteristics alone distinguish effective leaders?", 
        "precise personalities"
    ),
    (
        "Where did Bentley Motors establish a new facility to simultaneously improve technical workforce skills and develop future professional leaders?", 
        "affiliate college"
    ),
    (
        "In a business environment, what does the application of empathy explicitly NOT imply doing with other people's emotions?", 
        "espousing"
    ),
    (
        "According to the executive director of the Leadership Institute at the University of Southern California, why are many leadership training programs left without significance?", 
        "too superficial"
    ),
    (
        "According to Mittal and Sindhu, what specific mental exercise should managers do to regenerate new and energizing goal statements?", 
        "recall why they loved their career"
    ),
    (
        "Unlike basic communication skills, what specific leadership ability requires outright experience of several years and alignment of organizational objectives with personal goals?", 
        "visionary leadership"
    ),
    (
        "Modern effective management training programs concentrate heavily on business simulation and what other specific learning approach?", 
        "action learning"
    )
]

def evaluatehit(strategy_name: str, chunks: list, embed_chunks: np.ndarray) -> dict:
    hits = 0
    total_questions = len(qa)
    num_chunks = len(chunks)
    if num_chunks > 0:
        mean_len = int(sum([len(chunk) for chunk in chunks]) / num_chunks)
    else:
        mean_len = 0

    for query, expected_answer in qa:
        top_k_chunks = retrieve(query, chunks, embed_chunks, top_k=3)
        
        # Check if the expected answer is a substring in ANY of the top 3 chunks (case-insensitive)
        hit_found = any(expected_answer.lower() in chunk.lower() for chunk in top_k_chunks)
        
        if hit_found:
            hits += 1
            
    hit_rate = f"{hits}/{total_questions}"
    
    return {
        "Strategy": strategy_name,
        "Chunks": num_chunks,
        "Mean Len": f"{mean_len} chrs",
        "Hit Rate": hit_rate
    }


In [ ]:
results = [
    evaluatehit("Fixed-size", fixedchunks, embedfixed),
    evaluatehit("Sentence-based", sentencechunks, embedsentence),
    evaluatehit("Sliding window", slidingchunks, embedsliding)
]


print(f"{'Strategy':<16} | {'Chunks':<6} | {'Mean Len':<8} | {'Hit Rate'}")
print("-" * 50)
for r in results:
    print(f"{r['Strategy']:<16} | {r['Chunks']:<6} | {r['Mean Len']:<8} | {r['Hit Rate']}")

## Question 3

### Part A:Decision Tree


In [ ]:
def recommend_approach(scenario: dict) -> str:
    scale = scenario.get('scale') # 'prototype' or 'production'
    budget = scenario.get('budget')
    dynamic = scenario.get('data_changes_frequently')
    k_type = scenario.get('knowledge_type')
    out_format = scenario.get('need_specific_output_format')
    latency = scenario.get('latency_sensitivity')
    
    # 1. LATENCY OVERRIDE (Hard physical constraint)
    if latency:
        if dynamic:
            return "RAG + Fine-Tuning (Warning: RAG will inherently add some network latency)"
        return "Fine-Tuning"

    # 2. SCALE & BUDGET CHECK (The OpEx vs CapEx flip)
    if scale == 'production' and budget == 'low':
        # At high scale with low budget, you cannot afford prompt engineering token bloat.
        if dynamic:
            return "RAG (Optimized with minimal context)"
        return "Fine-Tuning (High upfront CapEx, but lowest OpEx for scale)"

    if scale == 'prototype' and budget == 'low':
        # At low scale, avoid all infrastructure costs.
        return "Prompt Engineering"

    # 3. ARCHITECTURE ROUTING (Medium/High Budgets)
    if dynamic or k_type == 'factual':
        # If data changes or relies on hard facts, RAG is mandatory.
        if out_format or k_type == 'both':
            return "RAG + Fine-Tuning"
        return "RAG"
    
    else:
        # Static data
        if out_format or k_type == 'behavioral':
            return "Fine-Tuning"
            
    # Default fallback for simple, static tasks with adequate budget
    return "Prompt Engineering"

In [ ]:
scenarios = [
    {
        "name": "Enterprise Support Agent",
        "scale": "production",
        "budget": "high",
        "data_changes_frequently": True,
        "knowledge_type": "both",
        "need_specific_output_format": True,
        "latency_sensitivity": True
    },
    {
        "name": "Live Market Tracker",
        "scale": "production",
        "budget": "medium",
        "data_changes_frequently": True,
        "knowledge_type": "factual",
        "need_specific_output_format": False,
        "latency_sensitivity": True
    },
    {
        "name": "Brand Persona Engine",
        "scale": "production",
        "budget": "high",
        "data_changes_frequently": False,
        "knowledge_type": "behavioral",
        "need_specific_output_format": True,
        "latency_sensitivity": False
    },
    {
        "name": "Student Prototype Tool",
        "scale": "prototype",
        "budget": "low",
        "data_changes_frequently": False,
        "knowledge_type": "factual",
        "need_specific_output_format": False,
        "latency_sensitivity": False
    },
    {
        "name": "Internal DB Query Builder",
        "scale": "production",
        "budget": "medium",
        "data_changes_frequently": True,
        "knowledge_type": "factual",
        "need_specific_output_format": True,
        "latency_sensitivity": False
    }
]

for s in scenarios:
    name = s.pop("name")
    print(f"Scenario: {name}")
    print(f"Result: {recommend_approach(s)}\n")

### Part B

In [ ]:
#Retrival functoinal slighlty modified to give the cosine_similarity value also for the top_k
def chunk_cosine_retrieval(query: str, chunks: list, embed_chunks: list, top_k: int = 3) -> list[str]:
    query_embed = model.encode(query)
    # similarities = [cosine_similarity(query_embed, chunkvector) for chunkvector in embed_chunks]
    # above is handwritten and takes longer for large lists
    similarities = (util.cos_sim(query_embed, embed_chunks)[0]).numpy()
    return [chunks[x] for x in np.argsort(similarities)[::-1][:top_k]],np.sort(similarities)[::-1][:top_k]

# Rag Pipeline from Question 1:
def querytest(query: str) -> str:
    retreived_chunks,cosines = chunk_cosine_retrieval(query, chunks, embed_chunks, top_k=3)
    context = "\n\n".join(chunk for chunk in retreived_chunks)
    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say 'I don't know'.
Context:
{context}
Question: {query}
Answer:"""
    response = generator(prompt)
    top100words = " ".join((context.split())[0:100])
    return [top100words, cosine_similarity(model.encode(query), model.encode(top100words)), response]

In [ ]:
def generatequerylogs(a : list[str], na : list[str]):
    labels = []
    scores = []
    colours = []

    print(f'Running Answerable Queries.\n{'--'*50}')
    for i,query in enumerate(a):
        chunk, cosine, answer = querytest(query)
        print(f'\nQuery A{i+1}: {query}')
        print(f'Top Retrieved Chunk : {chunk}')
        print(f'Similarity Score : {cosine:.5f}')
        print(f'Final Answer : {answer}')
        labels.append(f"A{i+1}")
        scores.append(cosine)
        colours.append("green")
        print('--'*50)

    print(f'\nRunning Non-Answerable Queries.\n{'--'*50}')
    for i,query in enumerate(na):
        chunk, cosine, answer = querytest(query)
        print(f'\nQuery N{i+1}: {query}')
        print(f'Top Retrieved Chunk : {chunk}')
        print(f'Similarity Score : {cosine:.5f}')
        print(f'Final Answer : {answer}')
        labels.append(f"N{i+1}")
        scores.append(cosine)
        if (answer.lower()).strip() == "i don't know" :
            colours.append("blue")
        else :
            colours.append("red")
        print('--'*50)
    return labels,scores,colours

In [ ]:
from matplotlib.patches import Patch

def generateplot(a : list[str], na : list[str]):
    labels, scores, colours = generatequerylogs(a,na)
    plt.figure(figsize=(10, 6))

    bars = plt.bar(labels, scores, color=colours, edgecolor='black')

    plt.title('RAG Benchmark: Cosine Similarity Scores by Query Type', fontsize=14, fontweight='bold')
    plt.xlabel('Queries', fontsize=12)
    plt.ylabel('Cosine Similarity Score', fontsize=12)

    # Ensure the Y-axis gives a realistic bounds for cosine similarity
    plt.ylim(0, max(scores) + 0.1 if scores else 1.0) 

    # Create custom legend
    legend_elements = [
        Patch(facecolor='green', edgecolor='black', label='Answerable (In-Scope)'),
        Patch(facecolor='blue', edgecolor='black', label='Out-Of-Scope (Answer:I don\'t know)'),
        Patch(facecolor='red', edgecolor='black', label='Out-Of-Scope (Had Hallucinated)')
    ]
    plt.legend(handles=legend_elements, loc='lower right')

    # Add score labels on top of each bar for exact data reading
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f'{yval:.2f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()
    pass


In [ ]:
# 16 queries strictly answerable using the provided document (In-Scope)
answerable_queries = [
    "How much capital is invested annually into training managers in the United Kingdom?",
    "According to Daniel Goleman's research, what are the five specific elements of emotional intelligence that determine a leader's effectiveness?",
    "Which specific leadership style is predominantly practiced in the armed forces, and from which theoretical framework does it originate?",
    "According to the two-year study conducted by Jay Conger, what five specific components are essential in the development of effective leaders?",
    "What was the specific outcome of the research conducted by the French pharmaceutical company Sanofi regarding its workforce?",
    "According to Mittal and Sindhu, what specific mental exercise should managers practice to regenerate energizing goal statements and improve self-motivation?",
    "What are the three specific leadership styles that originate from Kurt Lewin’s behavioural theories?",
    "In the study involving over 1800 army officers funded by the US Army and DoD, what specific type of leadership model was developed?",
    "According to Daniel Goleman, what specific practice can assist leaders in increasing their degree of self-awareness by spending a few minutes each day?",
    "What physical practice is recommended for managers to practice composure and avoid shouting at their team in challenging encounters?",
    "How does McCleskey define the concept of empathy in a leadership context?",
    "According to Daft, which specific element of emotional intelligence is considered the most treasured aspect of a leader?",
    "How many participants were involved in the two-year study conducted by Jay Conger regarding leadership development?",
    "Which historical philosopher believed that training a good leader required at least fifty years?",
    "What exact percentage of US executives surveyed believed that great leaders are born, rather than made or a combination of both?",
    "What specific feedback model is deemed the most efficient and can be integrated into all-inclusive management and leadership programmes?"
]

# 16 queries designed to trap the LLM (Out-of-Scope / Hallucination Traps)
unanswerable_queries = [
    "What specific metric did Google Inc. use to measure the return on investment (ROI) for its feedback-based management training programs?",
    "How does Daniel Goleman's emotional intelligence framework compare to the Myers-Briggs Type Indicator (MBTI) when assessing potential executive leaders?",
    "What were the exact percentage increases in employee retention at Bentley Motors after the establishment of their affiliate college training facility?",
    "Of the over 50 billion dollars spent annually on management training in the United Kingdom, what exact percentage is allocated exclusively to emotional intelligence workshops?",
    "How did the adoption of Kurt Lewin's Laissez-faire leadership style specifically impact the quarterly profit margins of Bentley Motors?",
    "What specific subjects or curriculum did Plato mandate during the fifty years he believed was required to train a good leader?",
    "What specific year did the US Army and Department of Defense publish their study involving over 1800 army officers?",
    "How many hours per week do managers at Google Inc. strictly dedicate to conducting peer reviews?",
    "What was the exact profit increase reported by Sanofi after implementing their emotional intelligence management training programs?",
    "According to Kurt Lewin's behavioural theories, what percentage of modern tech startups utilize a democratic leadership style?",
    "What is the name of the specific affiliate college that Bentley Motors partnered with to establish their Bentley Training Facility?",
    "How did Daniel Goleman's research address the impact of remote working environments on a leader's social skills?",
    "What specific psychological assessment tool did Mittal and Sindhu use to measure self-motivation in their 2012 study?",
    "In the survey of US executives regarding leadership origins, what was the total number of executives polled to gather the statistical data?",
    "What specific physical exercises, besides deep-breathing, did Karp (2015) recommend for improving self-regulation in high-stress meetings?",
    "How much does a standard commercial license for the 360 degrees feedback model software cost for mid-sized organizations?"
]

In [ ]:
generateplot(answerable_queries, unanswerable_queries)